In [1]:
import xarray as xr
import numpy as np

In [2]:
range_id = 3

silo = xr.open_dataset("/scratch/tm70/ag9761/iceds_datasets/silo/silo_monthly_rain.nc")
# silo = silo.assign_coords(dataset_name='silo')

/scratch/tm70/ag9761/conda/envs/iceds/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
silo

<xarray.Dataset> Size: 1GB
Dimensions:       (time: 288, lat: 681, lon: 841)
Coordinates:
  * time          (time) datetime64[ns] 2kB 1993-01-16T12:00:00 ... 2016-12-1...
  * lat           (lat) float64 5kB -44.0 -43.95 -43.9 ... -10.1 -10.05 -10.0
  * lon           (lon) float64 7kB 112.0 112.0 112.1 ... 153.9 153.9 154.0
Data variables:
    monthly_rain  (time, lat, lon) float64 1GB ...
Attributes: (12/14)
    CDI:                      Climate Data Interface version 2.4.3 (https://m...
    Conventions:              CF-1.6
    institution:              Queensland Government, Department of Environmen...
    department:               Department of Environment and Science
    department_short:         DES
    copyright:                Copyright - the State of Queensland Department ...
    ...                       ...
    raster_source_additions:  and other suppliers (see the SILO webpage for d...
    metadata_url:             http://qldspatial.information.qld.gov.au/catalo...
    reference:                Jeffrey, S.J., Carter, J.O., Moodie, K.B.A. and...
    disclaimer:               1. The user accepts all responsibility and risk...
    history:                  Wed Apr 22 17:32:41 2026: cdo mergetime 1993.mo...
    CDO:                      Climate Data Operators version 2.4.3 (https://m...

In [4]:
if range_id == 0:
    lat_range = (-25, -22)
    lon_range = (142, 145)
    radius = 1.15 # Setting the radius as the one from which different grid radii would be taken later
if range_id == 1:
    lat_range = (-29, -26)
    lon_range = (150, 153)
    radius = 1.15 # Setting the radius as the one from which different grid radii would be taken later
if range_id == 2:
    lat_range = (-20, -17)
    lon_range = (143, 146) 
    radius = 1.15 # Setting the radius as the one from which different grid radii would be taken later
if range_id == -1:
    lat_range = (-29, -17)
    lon_range = (142, 153)
    radius = 1.15 # Setting the radius as the one from which different grid radii would be taken later
if range_id == 3:
    lat_range = (-29, -10)
    lon_range = (138, 153)
    radius = 1.15 # Setting the radius as the one from which different grid radii would be taken later


In [5]:
surrounding_grid = np.array([-radius, 0, radius]) 
lat_range = (lat_range[0] - radius, lat_range[1] + radius + 1)
lon_range = (lon_range[0] - radius, lon_range[1] + radius + 1)


In [6]:
silo = silo.sel(lat=slice(*lat_range), lon=slice(*lon_range))

In [7]:
silo

<xarray.Dataset> Size: 320MB
Dimensions:       (time: 288, lat: 404, lon: 344)
Coordinates:
  * time          (time) datetime64[ns] 2kB 1993-01-16T12:00:00 ... 2016-12-1...
  * lat           (lat) float64 3kB -30.15 -30.1 -30.05 ... -10.1 -10.05 -10.0
  * lon           (lon) float64 3kB 136.8 136.9 136.9 ... 153.9 153.9 154.0
Data variables:
    monthly_rain  (time, lat, lon) float64 320MB ...
Attributes: (12/14)
    CDI:                      Climate Data Interface version 2.4.3 (https://m...
    Conventions:              CF-1.6
    institution:              Queensland Government, Department of Environmen...
    department:               Department of Environment and Science
    department_short:         DES
    copyright:                Copyright - the State of Queensland Department ...
    ...                       ...
    raster_source_additions:  and other suppliers (see the SILO webpage for d...
    metadata_url:             http://qldspatial.information.qld.gov.au/catalo...
    reference:                Jeffrey, S.J., Carter, J.O., Moodie, K.B.A. and...
    disclaimer:               1. The user accepts all responsibility and risk...
    history:                  Wed Apr 22 17:32:41 2026: cdo mergetime 1993.mo...
    CDO:                      Climate Data Operators version 2.4.3 (https://m...

In [8]:
# Take floor of that month
# TODO: Add test
silo = silo.resample(time="MS").mean()

In [9]:
silo = silo.convert_calendar("standard", use_cftime=True)

In [10]:
silo = silo.rename({"lat": "latitude", "lon": "longitude", "monthly_rain": "pr"})

In [11]:
# Convert to mm/day
gb = silo.groupby("time.days_in_month")
silo = gb.map(lambda x: x / x.time.dt.days_in_month)

In [12]:
silo["pr"] = silo["pr"].assign_attrs({"units" :  "mm / day"})

In [ ]:
# TODO: Based on hindcast/forecast change this line
silo.to_netcdf(f"/scratch/tm70/ag9761/iceds_datasets/silo_monthly_rain_iceds_{range_id}_processed_forecast.nc")

In [14]:
silo

<xarray.Dataset> Size: 320MB
Dimensions:    (time: 288, latitude: 404, longitude: 344)
Coordinates:
  * time       (time) object 2kB 1993-01-01 00:00:00 ... 2016-12-01 00:00:00
  * latitude   (latitude) float64 3kB -30.15 -30.1 -30.05 ... -10.1 -10.05 -10.0
  * longitude  (longitude) float64 3kB 136.8 136.9 136.9 ... 153.9 153.9 154.0
Data variables:
    pr         (time, latitude, longitude) float64 320MB 0.5065 0.5097 ... nan
Attributes: (12/16)
    CDI:                      Climate Data Interface version 2.4.3 (https://m...
    Conventions:              CF-1.6
    institution:              Queensland Government, Department of Environmen...
    department:               Department of Environment and Science
    department_short:         DES
    copyright:                Copyright - the State of Queensland Department ...
    ...                       ...
    reference:                Jeffrey, S.J., Carter, J.O., Moodie, K.B.A. and...
    disclaimer:               1. The user accepts all responsibility and risk...
    history:                  Wed Apr 22 17:32:41 2026: cdo mergetime 1993.mo...
    CDO:                      Climate Data Operators version 2.4.3 (https://m...
    standard_name:            time
    axis:                     T